# RNN Weather Prediction — LSTM vs GRU

**Task:** Predict **tomorrow's mean temperature** for a chosen city using the Zenodo weather dataset.

### Record questions
1. What is an RNN?
2. Explain vanishing/exploding gradients in RNNs.

### Required comparison
- With vs **without seasonality features**
- **LSTM vs GRU** with comparable parameters
- **MAE** and **RMSE** on validation and test sets

The notebook uses a chronological split to avoid future-data leakage.

## 1. What is an RNN?

A **Recurrent Neural Network (RNN)** is a neural network designed for sequential data. It maintains a hidden state that carries information from previous time steps.

A simple RNN can be written as:

`h_t = tanh(W_x x_t + W_h h_(t-1) + b)`

The hidden state summarizes information from the current input and earlier inputs. RNNs are therefore useful for time series, text, speech, and other sequential problems.

Here, a sequence of recent weather observations is used to predict the **next day's mean temperature**.

## 2. Vanishing and Exploding Gradients

During backpropagation through time, gradients are repeatedly multiplied through recurrent connections.

- **Vanishing gradients:** repeated multiplication makes gradients very small, so early time steps receive little learning signal. Long-term dependencies become difficult to learn.
- **Exploding gradients:** repeated multiplication makes gradients very large, producing unstable updates or divergence.

**LSTM** and **GRU** use gates to control information flow and reduce these problems. The models below also use gradient clipping for additional training stability.

In [ ]:
!pip -q install tensorflow pandas numpy scikit-learn matplotlib seaborn

import os, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

In [ ]:
URL = "https://zenodo.org/records/7525955/files/weather_prediction_dataset.csv?download=1"
DATA_FILE = "/content/weather_prediction_dataset.csv"

if not os.path.exists(DATA_FILE):
    !wget -q -O /content/weather_prediction_dataset.csv "$URL"

df = pd.read_csv(DATA_FILE)
print("Shape:", df.shape)
display(df.head())
print(df.columns.tolist())

In [ ]:
# Identify date column
date_candidates = [c for c in df.columns
                    if str(c).lower() in ["date","datetime","time","timestamp"]]
if not date_candidates:
    date_candidates = [c for c in df.columns
                        if any(k in str(c).lower() for k in ["date","time"])]
if not date_candidates:
    raise ValueError("Could not identify the date column.")

DATE_COL = date_candidates[0]
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)

# Find numeric temperature/mean-temperature columns
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
temp_candidates = [c for c in numeric_cols
                    if any(k in str(c).lower()
                           for k in ["mean","temp","temperature","tmean"])]

preferred = [c for c in temp_candidates if "mean" in str(c).lower()]
if not preferred:
    preferred = temp_candidates
if not preferred:
    raise ValueError("No numeric temperature column was detected.")

print("Date column:", DATE_COL)
print("Possible mean-temperature columns:")
for i,c in enumerate(preferred):
    print(i, "->", c)

# Automatically choose the first mean-temperature column.
TARGET_COL = preferred[0]
print("\nSelected target:", TARGET_COL)

### Choosing a city

The target column is selected automatically from the downloaded CSV. To use another city, change `TARGET_COL` in the previous cell after viewing the printed candidates.

For example:

```python
TARGET_COL = "Basel_mean"
```

Use the exact column name printed by the notebook.

In [ ]:
# Determine columns belonging to the selected city.
# Common dataset naming is City_variable, so the prefix before "_" is used.
city_prefix = str(TARGET_COL).split("_")[0]

city_cols = [
    c for c in df.columns
    if str(c).startswith(city_prefix + "_")
    and c != DATE_COL
    and pd.api.types.is_numeric_dtype(df[c])
]
if TARGET_COL not in city_cols:
    city_cols.append(TARGET_COL)

print("Selected city/prefix:", city_prefix)
print("Weather columns used:")
print(city_cols)

work = df[[DATE_COL] + city_cols].copy()
for c in city_cols:
    work[c] = pd.to_numeric(work[c], errors="coerce")

# Target at day t is tomorrow's mean temperature relative to input ending at t-1.
work["target"] = work[TARGET_COL].shift(-1)

# Annual cyclic seasonality features
doy = work[DATE_COL].dt.dayofyear
work["season_sin"] = np.sin(2*np.pi*doy/365.25)
work["season_cos"] = np.cos(2*np.pi*doy/365.25)

work = work.dropna(subset=["target"]).reset_index(drop=True)

print("Prepared rows:", len(work))
display(work.head())

## 3. Chronological train/validation/test split

- 70% training
- 15% validation
- 15% test

The data is **not randomly shuffled**. Feature and target scalers are fitted only on training observations.

In [ ]:
LOOKBACK = 14
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

n = len(work)
train_end = int(n * TRAIN_FRAC)
val_end = int(n * (TRAIN_FRAC + VAL_FRAC))

BASE_FEATURES = [c for c in city_cols if c != "target"]
SEASON_FEATURES = ["season_sin", "season_cos"]

print("Train rows:", train_end)
print("Validation rows:", val_end-train_end)
print("Test rows:", n-val_end)
print("Base features:", BASE_FEATURES)

In [ ]:
def make_sequences(feature_cols):
    data = work[[DATE_COL] + feature_cols + ["target"]].copy()

    # Fill missing weather values chronologically.
    X_raw = data[feature_cols].ffill().bfill().values.astype(np.float32)
    y_raw = data["target"].values.astype(np.float32)

    # Fit scalers ONLY on training rows.
    feature_scaler = StandardScaler()
    target_scaler = StandardScaler()
    feature_scaler.fit(X_raw[:train_end])
    target_scaler.fit(y_raw[:train_end].reshape(-1,1))

    X_scaled = feature_scaler.transform(X_raw).astype(np.float32)
    y_scaled = target_scaler.transform(y_raw.reshape(-1,1)).ravel().astype(np.float32)

    Xtr,ytr,dtr,Xv,yv,dv,Xte,yte,dte = [],[],[],[],[],[],[],[],[]

    # Input uses LOOKBACK days ending immediately before target day i.
    for i in range(LOOKBACK, len(data)):
        x = X_scaled[i-LOOKBACK:i]
        y = y_scaled[i]
        date = data[DATE_COL].iloc[i]

        if i < train_end:
            Xtr.append(x); ytr.append(y); dtr.append(date)
        elif i < val_end:
            Xv.append(x); yv.append(y); dv.append(date)
        else:
            Xte.append(x); yte.append(y); dte.append(date)

    return (np.array(Xtr),np.array(ytr),np.array(dtr),
            np.array(Xv),np.array(yv),np.array(dv),
            np.array(Xte),np.array(yte),np.array(dte),
            feature_scaler,target_scaler)

## 4. LSTM and GRU models

Both models use one recurrent layer with **32 units**, the same dropout and dense layers, optimizer, learning rate, batch size, and stopping rule.

This keeps the comparison controlled and their parameter counts in a comparable range.

In [ ]:
def build_lstm(input_shape, units=32):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(units),
        tf.keras.layers.Dropout(0.10),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

def build_gru(input_shape, units=32):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.GRU(units),
        tf.keras.layers.Dropout(0.10),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

demo_shape = (LOOKBACK, max(1,len(BASE_FEATURES)))
print("LSTM parameters:", build_lstm(demo_shape).count_params())
print("GRU parameters :", build_gru(demo_shape).count_params())

## 5. Train all four configurations

The four experiments are:

1. LSTM + no seasonality
2. LSTM + seasonality
3. GRU + no seasonality
4. GRU + seasonality

Early stopping restores the best validation-loss weights.

In [ ]:
EPOCHS = 40
BATCH_SIZE = 32
PATIENCE = 7

results = []
histories = {}
predictions = {}

def run_experiment(model_type, use_seasonality):
    features = BASE_FEATURES + (SEASON_FEATURES if use_seasonality else [])

    (Xtr,ytr,dtr,Xv,yv,dv,Xte,yte,dte,
     feature_scaler,target_scaler) = make_sequences(features)

    shape = (Xtr.shape[1], Xtr.shape[2])
    model = build_lstm(shape) if model_type=="LSTM" else build_gru(shape)

    callbacks = [tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=PATIENCE, restore_best_weights=True)]

    history = model.fit(
        Xtr,ytr, validation_data=(Xv,yv),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        shuffle=False, callbacks=callbacks, verbose=1
    )

    vp = model.predict(Xv, verbose=0).ravel()
    tp = model.predict(Xte, verbose=0).ravel()

    yv_true = target_scaler.inverse_transform(yv.reshape(-1,1)).ravel()
    yt_true = target_scaler.inverse_transform(yte.reshape(-1,1)).ravel()
    vp = target_scaler.inverse_transform(vp.reshape(-1,1)).ravel()
    tp = target_scaler.inverse_transform(tp.reshape(-1,1)).ravel()

    key = model_type + "_" + ("With_Seasonality" if use_seasonality else "Without_Seasonality")

    histories[key] = history.history
    predictions[key] = {
        "val_dates":dv, "val_true":yv_true, "val_pred":vp,
        "test_dates":dte, "test_true":yt_true, "test_pred":tp
    }

    results.append({
        "Model":model_type,
        "Seasonality":"With" if use_seasonality else "Without",
        "Parameters":model.count_params(),
        "Val MAE":mean_absolute_error(yv_true,vp),
        "Val RMSE":np.sqrt(mean_squared_error(yv_true,vp)),
        "Test MAE":mean_absolute_error(yt_true,tp),
        "Test RMSE":np.sqrt(mean_squared_error(yt_true,tp)),
        "Epochs":len(history.history["loss"])
    })

    print("\n",key)
    print("Parameters:",model.count_params())
    print("Val MAE:",results[-1]["Val MAE"])
    print("Val RMSE:",results[-1]["Val RMSE"])
    print("Test MAE:",results[-1]["Test MAE"])
    print("Test RMSE:",results[-1]["Test RMSE"])

for m in ["LSTM","GRU"]:
    for s in [False,True]:
        run_experiment(m,s)

In [ ]:
results_df = pd.DataFrame(results).sort_values("Test RMSE").reset_index(drop=True)

display(results_df.style.format({
    "Val MAE":"{:.4f}", "Val RMSE":"{:.4f}",
    "Test MAE":"{:.4f}", "Test RMSE":"{:.4f}"
}))

best = results_df.iloc[0]
print("\nBEST CONFIGURATION:")
print(best.to_string())

## 6. Error comparison

In [ ]:
plot_df = results_df.copy()
plot_df["Configuration"] = plot_df["Model"] + " — " + plot_df["Seasonality"]

fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.barplot(data=plot_df,x="Configuration",y="Val MAE",ax=axes[0])
axes[0].set_title("Validation MAE")
axes[0].tick_params(axis="x",rotation=25)

sns.barplot(data=plot_df,x="Configuration",y="Test MAE",ax=axes[1])
axes[1].set_title("Test MAE")
axes[1].tick_params(axis="x",rotation=25)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.barplot(data=plot_df,x="Configuration",y="Val RMSE",ax=axes[0])
axes[0].set_title("Validation RMSE")
axes[0].tick_params(axis="x",rotation=25)

sns.barplot(data=plot_df,x="Configuration",y="Test RMSE",ax=axes[1])
axes[1].set_title("Test RMSE")
axes[1].tick_params(axis="x",rotation=25)
plt.tight_layout()
plt.show()

## 7. Training convergence

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(14,9))
for ax,(key,h) in zip(axes.ravel(),histories.items()):
    ax.plot(h["loss"],label="Training Loss")
    ax.plot(h["val_loss"],label="Validation Loss")
    ax.set_title(key.replace("_"," "))
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend()
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Test-set actual vs predicted temperature

In [ ]:
for key,p in predictions.items():
    plt.figure(figsize=(14,4))
    plt.plot(p["test_dates"],p["test_true"],label="Actual",linewidth=2)
    plt.plot(p["test_dates"],p["test_pred"],label="Predicted",linewidth=1.5)
    plt.title(key.replace("_"," ") + " — Test Set")
    plt.xlabel("Date")
    plt.ylabel("Mean Temperature")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 9. Conclusion for the record

After running the notebook, use the generated comparison table to report the exact **validation MAE, validation RMSE, test MAE, and test RMSE**.

A suitable conclusion is:

> LSTM and GRU models were implemented to forecast tomorrow's mean temperature. The models were compared with and without explicit annual seasonality features. Chronological train/validation/test splits were used to prevent temporal leakage, and scaling was fitted only on the training data. MAE and RMSE were used as evaluation metrics. The best model is the configuration with the lowest test error while maintaining comparable validation performance. The effect of seasonality can be determined by comparing each architecture with and without the seasonal sine/cosine features.

In [ ]:
# Save final results for the lab record
results_df.to_csv("/content/rnn_weather_results.csv", index=False)
print("Saved /content/rnn_weather_results.csv")